# Projetinho — Surrogate Model + Otimização Bayesiana
## Módulo: Inteligência Artificial (Machine Learning) — Aplicação Naval

**Dataset:** Yacht Hydrodynamics (Delft Ship Hydromechanics Laboratory, via UCI ML Repository) — 308 ensaios de resistência residual de cascos de veleiro, em função de 5 parâmetros geométricos de forma + número de Froude.

**Objetivo do exercício:** treinar um *surrogate model* (Gaussian Process) sobre os dados reais e usá-lo como "função cara" de referência para demonstrar **Otimização Bayesiana** buscando a combinação de parâmetros de casco que minimiza a resistência residual — comparando com uma busca aleatória de mesmo orçamento de avaliações.

Roda inteiramente em CPU, sem dataset externo pesado, tempo total esperado: poucos minutos.

## 1. Instalação e imports

In [ ]:
# No Google Colab, só o ucimlrepo normalmente precisa ser instalado
!pip install -q ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
import warnings; warnings.filterwarnings("ignore")  # deixa a saída mais limpa em aula

RNG = np.random.default_rng(42)

## 2. Carregar o dataset (Yacht Hydrodynamics — UCI)

In [ ]:
from ucimlrepo import fetch_ucirepo

yacht = fetch_ucirepo(id=243)
X_full = yacht.data.features.copy()
y_full = yacht.data.targets.copy()

# Renomeia colunas para nomes mais legíveis
X_full.columns = ["LCB", "Cp", "L_disp", "B_T", "L_B", "Froude"]
y_full = y_full.iloc[:, 0]
y_full.name = "Rr"

df = pd.concat([X_full, y_full], axis=1)
print(df.shape)
df.head()

**Colunas:**
- `LCB` — posição longitudinal do centro de flutuação (adimensional)
- `Cp` — coeficiente prismático
- `L_disp` — razão comprimento/deslocamento
- `B_T` — razão boca/calado
- `L_B` — razão comprimento/boca
- `Froude` — número de Froude (velocidade adimensional)
- `Rr` — resistência residual por unidade de peso de deslocamento (alvo)

Os 5 primeiros parâmetros descrevem **22 formas de casco diferentes**; cada uma foi ensaiada em 14 velocidades (Froude). É isso que vamos tratar como nosso "espaço de projeto".

## 3. Passo 1 — Surrogate "verdade-base" (ground truth)

Ajustamos um Gaussian Process sobre **todo o dataset real** (5 parâmetros geométricos + Froude). Esse modelo vai fazer o papel da "simulação cara" (no lugar de um CFD de verdade) — é contra ele que o loop de otimização Bayesiana vai consultar, ponto a ponto, como se cada avaliação custasse caro.

In [ ]:
feature_cols = ["LCB", "Cp", "L_disp", "B_T", "L_B", "Froude"]
X = df[feature_cols].values
y = df["Rr"].values

x_scaler = StandardScaler().fit(X)
y_mean, y_std = y.mean(), y.std()

Xs = x_scaler.transform(X)
ys = (y - y_mean) / y_std

kernel_truth = (ConstantKernel(1.0, (1e-2, 1e2))
                * RBF(length_scale=np.ones(Xs.shape[1]), length_scale_bounds=(1e-2, 1e2))
                + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e-1)))

gp_truth = GaussianProcessRegressor(kernel=kernel_truth, n_restarts_optimizer=5, normalize_y=False)
gp_truth.fit(Xs, ys)

r2 = gp_truth.score(Xs, ys)
print(f"R² do surrogate 'verdade-base' nos dados de treino: {r2:.4f}")

## 4. Espaço de projeto: fixar a velocidade (Froude) e restringir aos cascos realmente testados

Vamos fixar o número de Froude num valor de operação intermediário (Fn = 0.30) e buscar, entre os 5 parâmetros geométricos, a combinação que **minimiza a resistência residual**.

**Cuidado importante:** o dataset tem só **22 combinações reais** de casco (cada uma ensaiada em 14 velocidades). Usar o mínimo/máximo de cada parâmetro *isoladamente* como limite não é suficiente — um ponto pode respeitar cada limite individual e, ainda assim, ser uma combinação dos 5 parâmetros que nunca foi testada em conjunto (extrapolação no espaço conjunto). Para evitar isso, os candidatos avaliados serão sempre **combinações convexas dos 22 cascos reais** (uma média ponderada entre designs existentes) — isso garante geometricamente que todo candidato fica dentro do fecho convexo dos dados observados, ou seja, sempre em regime de interpolação.

In [ ]:
design_cols = ["LCB", "Cp", "L_disp", "B_T", "L_B"]
FIXED_FROUDE = 0.30

# As 22 combinações reais de casco (ignorando a duplicação por Froude)
unique_hulls = df[design_cols].drop_duplicates().values
n_hulls = unique_hulls.shape[0]
print(f"Número de cascos (formas) distintos no dataset: {n_hulls}")

def expensive_eval(design_params):
    """Simula uma avaliação 'cara' (ex.: CFD) consultando o surrogate verdade-base.
    design_params: array (5,) na ordem de design_cols.
    Retorna a resistência residual prevista (escala original)."""
    x = np.concatenate([design_params, [FIXED_FROUDE]]).reshape(1, -1)
    xs = x_scaler.transform(x)
    pred_s = gp_truth.predict(xs)[0]
    return pred_s * y_std + y_mean

def sample_convex_designs(n, alpha=1.0, rng=None):
    """Amostra n designs como combinações convexas dos 22 cascos reais.
    alpha baixo (< 1) tende a gerar pesos mais concentrados em poucos cascos
    (candidatos mais próximos de designs reais); alpha alto tende à média geral."""
    if rng is None:
        rng = np.random.default_rng()
    weights = rng.dirichlet(alpha * np.ones(n_hulls), size=n)  # (n, n_hulls), cada linha soma 1
    return weights @ unique_hulls  # (n, 5) — sempre dentro do fecho convexo real

## 5. Passo 2 — Otimização Bayesiana "na mão" (surrogate leve + Expected Improvement)

Aqui está o núcleo do exercício: em vez de usar uma biblioteca pronta, implementamos o loop de Otimização Bayesiana de forma explícita, para deixar visível cada peça:

1. Poucas avaliações iniciais (orçamento caro, tipo "rodar CFD 5 vezes") — sempre amostradas como combinações convexas dos 22 cascos reais
2. Um **surrogate leve** (outro GP, agora só sobre os pontos já avaliados) é reajustado a cada iteração
3. A função de aquisição **Expected Improvement (EI)** é avaliada só sobre candidatos igualmente restritos ao fecho convexo dos cascos reais
4. Repete até esgotar o orçamento de avaliações

A busca aleatória de comparação usa a mesma restrição — assim a comparação entre os dois métodos é justa e nenhum dos dois "trapaceia" extrapolando.

In [ ]:
def expected_improvement(X_cand, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X_cand, return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    imp = y_best - mu - xi          # minimização: queremos mu menor que y_best
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-9] = 0.0
    return ei


def bayesian_optimization(n_init=5, n_iter=15, n_candidates=2000, seed=0):
    rng = np.random.default_rng(seed)

    X_obs = sample_convex_designs(n_init, alpha=1.0, rng=rng)
    y_obs = np.array([expensive_eval(x) for x in X_obs])
    best_so_far = [y_obs.min()]

    x_scaler_bo = StandardScaler()

    for it in range(n_iter):
        Xs_obs = x_scaler_bo.fit_transform(X_obs)
        ys_mean_bo, ys_std_bo = y_obs.mean(), y_obs.std() + 1e-9
        ys_obs = (y_obs - ys_mean_bo) / ys_std_bo

        kernel_bo = (ConstantKernel(1.0, (1e-2, 1e2))
                     * RBF(length_scale=np.ones(len(design_cols)), length_scale_bounds=(1e-2, 1e2))
                     + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-6, 1e0)))
        gp_bo = GaussianProcessRegressor(kernel=kernel_bo, n_restarts_optimizer=3)
        gp_bo.fit(Xs_obs, ys_obs)

        # Candidatos também restritos ao fecho convexo dos 22 cascos reais
        X_cand = sample_convex_designs(n_candidates, alpha=0.5, rng=rng)
        Xs_cand = x_scaler_bo.transform(X_cand)
        y_best_scaled = (y_obs.min() - ys_mean_bo) / ys_std_bo
        ei = expected_improvement(Xs_cand, gp_bo, y_best_scaled)

        x_next = X_cand[np.argmax(ei)]
        y_next = expensive_eval(x_next)

        X_obs = np.vstack([X_obs, x_next])
        y_obs = np.append(y_obs, y_next)
        best_so_far.append(y_obs.min())

    return X_obs, y_obs, np.array(best_so_far)


def random_search(n_total=20, seed=0):
    rng = np.random.default_rng(seed)
    X_obs = sample_convex_designs(n_total, alpha=1.0, rng=rng)
    y_obs = np.array([expensive_eval(x) for x in X_obs])
    best_so_far = np.minimum.accumulate(y_obs)
    return X_obs, y_obs, best_so_far


N_INIT, N_ITER = 5, 15
X_bo, y_bo, best_bo = bayesian_optimization(n_init=N_INIT, n_iter=N_ITER, seed=1)
X_rs, y_rs, best_rs = random_search(n_total=N_INIT + N_ITER, seed=1)

print(f"Melhor Rr encontrado — Otimização Bayesiana: {best_bo[-1]:.4f}")
print(f"Melhor Rr encontrado — Busca Aleatória:       {best_rs[-1]:.4f}")

## 6. Convergência: Otimização Bayesiana × Busca Aleatória

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(range(1, len(best_bo) + 1), best_bo, marker="o", label="Otimização Bayesiana")
plt.plot(range(1, len(best_rs) + 1), best_rs, marker="s", label="Busca Aleatória")
plt.xlabel("Número de avaliações 'caras'")
plt.ylabel("Melhor Rr encontrado até então")
plt.title(f"Convergência — casco a Fn = {FIXED_FROUDE}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Melhor design de casco encontrado

In [ ]:
best_idx = np.argmin(y_bo)
best_design = X_bo[best_idx]

print(f"Menor Rr previsto: {y_bo[best_idx]:.4f}  (Froude fixo = {FIXED_FROUDE})\n")
print("Parâmetros de casco correspondentes (combinação convexa de designs reais):")
for name, val in zip(design_cols, best_design):
    print(f"  {name:8s} = {val:8.4f}")

## Notas para a aula / discussão com os alunos

- O GP "verdade-base" (`gp_truth`) faz aqui o papel de um solver caro (CFD); na prática, cada `expensive_eval` seria uma simulação de horas.
- O segundo GP (`gp_bo`), reajustado a cada iteração só com os pontos já avaliados, é o **surrogate real do processo de otimização** — é ele quem "não sabe" o resultado antes de avaliar.
- A restrição a combinações convexas dos 22 cascos reais é o que garante que a otimização nunca peça ao surrogate uma extrapolação fora do regime em que ele foi treinado — vale discutir com a turma por que isso importa mais aqui do que em outros módulos (o dataset é uma amostragem discreta de poucos designs, não um grid contínuo).
- Vale pedir para os alunos variarem `FIXED_FROUDE` (ex.: 0.15 vs 0.40) e observarem como o casco ótimo muda — isso ilustra por que "casco ótimo" depende do regime de operação.
- Extensão natural (sem aumentar muito o custo): trocar Expected Improvement por Lower Confidence Bound (LCB) e comparar o comportamento de exploração vs. explotação.
- Extensão para discussão: variar o parâmetro `alpha` da distribuição de Dirichlet e observar como isso muda a diversidade dos candidatos (alpha baixo → candidatos próximos de cascos reais específicos; alpha alto → candidatos próximos da média geral).